In [1]:
import os
import subprocess

print(f"Working directory: {os.getcwd()}")


result = subprocess.run(
    ["docker", "--version"],
    capture_output=True, text=True
)
print(f"Docker: {result.stdout.strip()}")

required_files = [
    "headline_embeddings.npy",
    "metadata.json",
    "embedding_pipeline.py",
    "semantic_search.py",
    "week3_search_baseline.json",
    "day4_verified.csv"
]

print("\nChecking Week 3 files:\n")
all_exist = True
for file in required_files:
    exists = os.path.exists(file)
    if not exists:
        all_exist = False
    print(f"{'EXISTS' if exists else 'MISSING'} — {file}")

print()
if all_exist:
    print("All Week 3 files present ")
    print("Ready to start Week 4")
else:
    print("Some files missing — check before continuing")

Working directory: c:\Users\RIDDHI ASHAR\Desktop\financial-intelligence-engine\notebook
Docker: Docker version 29.6.1, build 8900f1d

Checking Week 3 files:

EXISTS — headline_embeddings.npy
EXISTS — metadata.json
EXISTS — embedding_pipeline.py
EXISTS — semantic_search.py
EXISTS — week3_search_baseline.json
EXISTS — day4_verified.csv

All Week 3 files present 
Ready to start Week 4


In [2]:
import subprocess
import time
import os

print("Pulling Qdrant Docker image...")
print("This may take 2-3 minutes on first run (~250MB)\n")

result = subprocess.run(
    ["docker", "pull", "qdrant/qdrant"],
    capture_output=True,
    text=True,
    timeout=300
)

print(result.stdout)
if result.returncode == 0:
    print("Qdrant image pulled successfully ")
else:
    print("Error pulling image:")
    print(result.stderr)

Pulling Qdrant Docker image...
This may take 2-3 minutes on first run (~250MB)

Using default tag: latest
latest: Pulling from qdrant/qdrant
4f4fb700ef54: Pulling fs layer
4f4fb700ef54: Pulling fs layer
4f4fb700ef54: Pulling fs layer
4f4fb700ef54: Pulling fs layer
4f4fb700ef54: Pulling fs layer
66db01a4699b: Pulling fs layer
5a90c418a99e: Pulling fs layer
70b841042f14: Pulling fs layer
1e00c2239a87: Pulling fs layer
4f4fb700ef54: Pulling fs layer
a57ddb0a7f09: Pulling fs layer
062e450697fa: Pulling fs layer
97903e4644f8: Pulling fs layer
4f4fb700ef54: Pulling fs layer
4f4fb700ef54: Download complete
5a90c418a99e: Download complete
a57ddb0a7f09: Download complete
70b841042f14: Download complete
27cee752998a: Download complete
97903e4644f8: Download complete
1e00c2239a87: Download complete
66db01a4699b: Download complete
15104bbcf718: Download complete
062e450697fa: Download complete
062e450697fa: Pull complete
97903e4644f8: Pull complete
5a90c418a99e: Pull complete
a57ddb0a7f09: Pull co

In [ ]:

subprocess.run(
    ["docker", "rm", "-f", "qdrant_financial"],
    capture_output=True, text=True
)


result = subprocess.run(
    [
        "docker", "run", "-d",
        "--name", "qdrant_financial",
        "-p", "6333:6333",
        "-p", "6334:6334",
        "-v", f"{os.getcwd()}/qdrant_storage:/qdrant/storage",
        "qdrant/qdrant"
    ],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    container_id = result.stdout.strip()
    print(f"Qdrant container started ")
    print(f"Container ID: {container_id[:12]}")
    print(f"REST API:     http://localhost:6333")
    print(f"Web UI:       http://localhost:6333/dashboard")
else:
    print("Error starting container:")
    print(result.stderr)

Qdrant container started 
Container ID: 9c49e32f6f93
REST API:     http://localhost:6333
Web UI:       http://localhost:6333/dashboard


In [5]:
import time
import urllib.request
import urllib.error

print("Waiting for Qdrant to be ready...")

max_attempts = 30
for attempt in range(max_attempts):
    try:
        response = urllib.request.urlopen(
            "http://localhost:6333/healthz",
            timeout=2
        )
        if response.status == 200:
            print(f"Qdrant is ready  (took {attempt+1} attempts)")
            break
    except Exception:
        time.sleep(1)
        if attempt % 5 == 0:
            print(f"  Still waiting... ({attempt+1}/{max_attempts})")
else:
    print("Qdrant did not start in time — check Docker logs")

Waiting for Qdrant to be ready...
Qdrant is ready  (took 1 attempts)


In [6]:
result = subprocess.run(
    ["docker", "ps", "--filter", "name=qdrant_financial"],
    capture_output=True, text=True
)
print(result.stdout)

result = subprocess.run(
    ["docker", "logs", "--tail", "5", "qdrant_financial"],
    capture_output=True, text=True
)
print("Last 5 log lines:")
print(result.stdout)
print(result.stderr)

CONTAINER ID   IMAGE           COMMAND             CREATED         STATUS         PORTS                                                             NAMES
9c49e32f6f93   qdrant/qdrant   "./entrypoint.sh"   2 minutes ago   Up 2 minutes   0.0.0.0:6333-6334->6333-6334/tcp, [::]:6333-6334->6333-6334/tcp   qdrant_financial

Last 5 log lines:
2026-07-21T02:31:55.024962Z  INFO actix_server::builder: starting 11 workers
2026-07-21T02:31:55.024968Z  INFO actix_server::server: Actix runtime found; starting in Actix runtime
2026-07-21T02:31:55.024970Z  INFO actix_server::server: starting service: "actix-web-service-0.0.0.0:6333", workers: 11, listening on: 0.0.0.0:6333
2026-07-21T02:31:55.029263Z  INFO qdrant::tonic: Qdrant gRPC listening on 6334
2026-07-21T02:31:55.029294Z  INFO qdrant::tonic: TLS disabled for gRPC API




In [7]:
result = subprocess.run(
    ["pip", "install", "qdrant-client"],
    capture_output=True, text=True
)
print(result.stdout[-300:] if len(result.stdout) > 300 else result.stdout)
print("qdrant-client installation complete")

- 10/11 [qdrant-client]
   ---------------------------------------- 11/11 [qdrant-client]


qdrant-client installation complete


In [9]:
import qdrant_client
from qdrant_client import QdrantClient
print(f"qdrant-client imported successfully")
print(f"Ready to connect to Qdrant")

qdrant-client imported successfully
Ready to connect to Qdrant


In [10]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams


client = QdrantClient(
    host="localhost",
    port=6333
)

collections = client.get_collections()
print(f"Connected to Qdrant ")
print(f"Existing collections: {collections}")


Connected to Qdrant 
Existing collections: collections=[]


In [12]:
COLLECTION_NAME = "financial_headlines"
VECTOR_SIZE     = 384

try:
    client.delete_collection(COLLECTION_NAME)
    print(f"Deleted existing collection: {COLLECTION_NAME}")
except Exception:
    pass

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=VECTOR_SIZE,
        distance=Distance.COSINE
    )
)

print(f"Collection created: {COLLECTION_NAME} ")
print(f"Vector size:        {VECTOR_SIZE} dimensions")
print(f"Distance metric:    Cosine similarity")


collection_info = client.get_collection(COLLECTION_NAME)
print(f"\nCollection info:")
print(f"  Status:  {collection_info.status}")
print(f"  Config:  {collection_info.config.params.vectors}")

Deleted existing collection: financial_headlines
Collection created: financial_headlines 
Vector size:        384 dimensions
Distance metric:    Cosine similarity

Collection info:
  Status:  green
  Config:  size=384 distance=<Distance.COSINE: 'Cosine'> hnsw_config=None quantization_config=None on_disk=None datatype=None multivector_config=None


In [13]:
print("=== Day 1 Summary ===\n")


result = subprocess.run(
    ["docker", "ps", "--filter", "name=qdrant_financial",
     "--format", "{{.Status}}"],
    capture_output=True, text=True
)
container_status = result.stdout.strip()


collection_info = client.get_collection(COLLECTION_NAME)

print(f"Docker container:  {container_status}")
print(f"Qdrant URL:        http://localhost:6333")
print(f"Collection name:   {COLLECTION_NAME}")
print(f"Collection status: {collection_info.status}")
print(f"Vector dimensions: {VECTOR_SIZE}")
print(f"Distance metric:   Cosine")
print(f"Points uploaded:   0 (Day 2 uploads all 9380)")
print(f"\nCarries into Day 2:")
print(f"  client object     → upload embeddings")
print(f"  COLLECTION_NAME   → {COLLECTION_NAME}")
print(f"  VECTOR_SIZE       → {VECTOR_SIZE}")

=== Day 1 Summary ===

Docker container:  Up 13 minutes
Qdrant URL:        http://localhost:6333
Collection name:   financial_headlines
Collection status: green
Vector dimensions: 384
Distance metric:   Cosine
Points uploaded:   0 (Day 2 uploads all 9380)

Carries into Day 2:
  client object     → upload embeddings
  COLLECTION_NAME   → financial_headlines
  VECTOR_SIZE       → 384


DAY 2- uploading Embeddings to Qdrant


In [1]:
import os
import json
import time
import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct,
    Filter,
    FieldCondition,
    MatchValue
)

COLLECTION_NAME = "financial_headlines"
VECTOR_SIZE = 384
BATCH_SIZE = 100
LABEL_MAP = {0 : "Bearish", 1:"Bullish", 2: "Neutral"}

print("All imports successful")
print(f"Working directory: {os.getcwd()}")

All imports successful
Working directory: c:\Users\RIDDHI ASHAR\Desktop\financial-intelligence-engine\notebook


In [2]:

embeddings_matrix = np.load("headline_embeddings.npy")
print(f"Embeddings loaded: {embeddings_matrix.shape}")


with open("metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)
print(f"Metadata loaded:   {len(metadata)} entries")

assert len(metadata) == len(embeddings_matrix), \
    "Embeddings and metadata count mismatch"
print(f"Count match:       {len(metadata) == len(embeddings_matrix)} ")

Embeddings loaded: (9380, 384)
Metadata loaded:   9380 entries
Count match:       True 


In [3]:
client = QdrantClient(host= "localhost",port = 6333)
collections = client.get_collections()
print(f"Connected to Qdrant ")
print(f"Collections: {[c.name for c in collections.collections]}")

collection_info = client.get_collection(COLLECTION_NAME)
print(f"Collection status : {collection_info.status}")

Connected to Qdrant 
Collections: ['financial_headlines']
Collection status : green


In [4]:
def prepare_points(embeddings, metadata):
    """
    Convert embeddings and metadata into Qdrant PointStruct objects.
    Each point has id, vector and payload.
    """
    points = []
    for i, (emb, meta) in enumerate(zip(embeddings, metadata)):
        point = PointStruct(
            id      = meta["id"],
            vector  = emb.tolist(),
            payload = {
                "original_text": meta["original_text"],
                "cleaned_text":  meta["cleaned_text"],
                "ticker":        meta["ticker"],
                "tickers_list":  meta["tickers_list"],
                "label":         meta["label"],
                "sentiment":     meta["sentiment"],
                "word_count":    meta["word_count"],
                "token_count":   meta["token_count"]
            }
        )
        points.append(point)
    return points

# test on first 3 points
sample_points = prepare_points(
    embeddings_matrix[:3],
    metadata[:3]
)

print("=== Sample Point Structure ===\n")
for p in sample_points:
    print(f"ID:      {p.id}")
    print(f"Vector:  [{p.vector[0]:.4f}, {p.vector[1]:.4f}, ...]"
          f" ({len(p.vector)} dims)")
    print(f"Payload: {p.payload}")
    print()

=== Sample Point Structure ===

ID:      0
Vector:  [-0.1035, -0.0058, ...] (384 dims)
Payload: {'original_text': '$BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT', 'cleaned_text': '$BYND - jpmorgan reels in expectations on beyond meat', 'ticker': '$BYND', 'tickers_list': ['$BYND'], 'label': 0, 'sentiment': 'Bearish', 'word_count': 10, 'token_count': 9}

ID:      1
Vector:  [-0.0017, 0.0575, ...] (384 dims)
Payload: {'original_text': '$CCL $RCL - Nomura points to bookings weakness at Carnival and Royal Caribbean https://t.co/yGjpT2ReD3', 'cleaned_text': '$CCL $RCL - nomura points to bookings weakness at carnival and royal caribbean', 'ticker': '$RCL,$CCL', 'tickers_list': ['$RCL', '$CCL'], 'label': 0, 'sentiment': 'Bearish', 'word_count': 14, 'token_count': 13}

ID:      2
Vector:  [-0.0669, 0.0356, ...] (384 dims)
Payload: {'original_text': '$CX - Cemex cut at Credit Suisse, J.P. Morgan on weak building outlook https://t.co/KN1g4AWFIb', 'cleaned_text': '$

In [5]:
print(f"Starting batch upload to Qdrant...")
print(f"Total points:  {len(metadata)}")
print(f"Batch size:    {BATCH_SIZE}")
print(f"Total batches: {len(metadata) // BATCH_SIZE + 1}\n")

start_time = time.time()
total_uploaded = 0

for i in range(0, len(metadata), BATCH_SIZE):
   
    batch_embeddings = embeddings_matrix[i:i + BATCH_SIZE]
    batch_metadata   = metadata[i:i + BATCH_SIZE]
    batch_points     = prepare_points(batch_embeddings, batch_metadata)

    
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=batch_points
    )

    total_uploaded += len(batch_points)

    
    if (i // BATCH_SIZE + 1) % 10 == 0 or \
       i + BATCH_SIZE >= len(metadata):
        elapsed = time.time() - start_time
        print(f"  Uploaded {total_uploaded}/{len(metadata)} points "
              f"({total_uploaded/len(metadata)*100:.1f}%) "
              f"— {elapsed:.1f}s elapsed")

total_time = time.time() - start_time
print(f"\nUpload complete!")
print(f"Total time:    {total_time:.1f} seconds")
print(f"Total points:  {total_uploaded}")

Starting batch upload to Qdrant...
Total points:  9380
Batch size:    100
Total batches: 94

  Uploaded 1000/9380 points (10.7%) — 1.7s elapsed
  Uploaded 2000/9380 points (21.3%) — 2.7s elapsed
  Uploaded 3000/9380 points (32.0%) — 3.7s elapsed
  Uploaded 4000/9380 points (42.6%) — 4.8s elapsed
  Uploaded 5000/9380 points (53.3%) — 5.6s elapsed
  Uploaded 6000/9380 points (64.0%) — 6.6s elapsed
  Uploaded 7000/9380 points (74.6%) — 7.8s elapsed
  Uploaded 8000/9380 points (85.3%) — 8.7s elapsed
  Uploaded 9000/9380 points (95.9%) — 10.1s elapsed
  Uploaded 9380/9380 points (100.0%) — 10.6s elapsed

Upload complete!
Total time:    10.6 seconds
Total points:  9380


In [6]:
print("=== Point Count Verification ===\n")

time.sleep(2)

collection_info = client.get_collection(COLLECTION_NAME)
print(f"Collection:     {COLLECTION_NAME}")
print(f"Status:         {collection_info.status}")
print(f"Points in DB:   {collection_info.points_count}")
print(f"Expected:       {len(metadata)}")
print(f"Match:          {collection_info.points_count == len(metadata)}")

if collection_info.points_count == len(metadata):
    print(f"\nAll {len(metadata)} points uploaded correctly ")
else:
    diff = len(metadata) - collection_info.points_count
    print(f"\nMissing {diff} points — rerun upload cell")

=== Point Count Verification ===

Collection:     financial_headlines
Status:         green
Points in DB:   9380
Expected:       9380
Match:          True

All 9380 points uploaded correctly 


In [7]:
print("=== Vector Verification ===\n")
test_id = 0 
results = client.retrieve(
    collection_name=COLLECTION_NAME,
    ids=[test_id],
    with_payload=False,
    with_vectors=True

)

if results:
    stored_vector = np.array(results[0].vector)
    original_vector = embeddings_matrix[test_id]
    match = np.allclose(stored_vector, original_vector)
    print(f"Point ID:         {test_id}")
    print(f"Stored vector:    [{stored_vector[0]:.4f}, "
          f"{stored_vector[1]:.4f}, ...] ({len(stored_vector)} dims)")
    print(f"Original vector:  [{original_vector[0]:.4f}, "
          f"{original_vector[1]:.4f}, ...] ({len(original_vector)} dims)")
    print(f"Vectors match:    {'PASS' if match else 'FAIL'}")

=== Vector Verification ===

Point ID:         0
Stored vector:    [-0.1035, -0.0058, ...] (384 dims)
Original vector:  [-0.1035, -0.0058, ...] (384 dims)
Vectors match:    PASS


In [8]:
from qdrant_client.models import ScoredPoint
from sentence_transformers import SentenceTransformer

print("=== Basic Search Test ===\n")


embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

query = "Tesla beats earnings expectations"
query_vector = embedding_model.encode(
    query, convert_to_numpy=True
).tolist()


start = time.time()
search_results = client.search(
    collection_name=COLLECTION_NAME,
    query_vector=query_vector,
    limit=5,
    with_payload=True
)
latency = (time.time() - start) * 1000

print(f"Query:   {query}")
print(f"Latency: {latency:.2f}ms\n")

for i, result in enumerate(search_results):
    print(f"Rank {i+1} — Score: {result.score:.4f}")
    print(f"  Headline:  {result.payload['original_text'][:70]}")
    print(f"  Sentiment: {result.payload['sentiment']}")
    print(f"  Ticker:    {result.payload['ticker']}")
    print()

c:\Users\RIDDHI ASHAR\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== Basic Search Test ===



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4804.79it/s]


AttributeError: 'QdrantClient' object has no attribute 'search'

In [9]:
import time
from sentence_transformers import SentenceTransformer

print("=== Basic Search Test ===\n")

# Load embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# User query
query = "Tesla beats earnings expectations"

# Convert query into 384-dimensional embedding
query_vector = embedding_model.encode(
    query,
    convert_to_numpy=True
).tolist()

# Search in Qdrant
start = time.time()

response = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=5,
    with_payload=True
)

latency = (time.time() - start) * 1000

# Extract search results
search_results = response.points

print(f"Query:   {query}")
print(f"Latency: {latency:.2f} ms\n")

for i, result in enumerate(search_results):
    print(f"Rank {i+1} — Score: {result.score:.4f}")
    print(f"  Headline:  {result.payload['original_text'][:70]}")
    print(f"  Sentiment: {result.payload['sentiment']}")
    print(f"  Ticker:    {result.payload['ticker']}")
    print()

=== Basic Search Test ===



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9314.65it/s]


Query:   Tesla beats earnings expectations
Latency: 274.22 ms

Rank 1 — Score: 0.6294
  Headline:  Tesla's stock surges 8.8% to top the $800 level
  Sentiment: Bullish
  Ticker:    

Rank 2 — Score: 0.6158
  Headline:  Tesla's Return to Profitability Is Not Sustainable
  Sentiment: Bearish
  Ticker:    

Rank 3 — Score: 0.5829
  Headline:  Tesla stock is wrapping up its best month since 2013, adding $40 billi
  Sentiment: Bullish
  Ticker:    

Rank 4 — Score: 0.5686
  Headline:  Money manager says she has 'very high' confidence Tesla shares reach $
  Sentiment: Bullish
  Ticker:    

Rank 5 — Score: 0.5446
  Headline:  $TSLA - Why Tesla Remains A Strong Buy. https://t.co/LyHcrFPlEH #marke
  Sentiment: Bullish
  Ticker:    $TSLA



In [10]:
print("=== Day 2 Complete Summary ===\n")

collection_info = client.get_collection(COLLECTION_NAME)

print(f"Collection:      {COLLECTION_NAME}")
print(f"Status:          {collection_info.status}")
print(f"Points uploaded: {collection_info.points_count}")
print(f"Vector size:     {VECTOR_SIZE} dimensions")
print(f"Distance:        Cosine")
print(f"\nVerification:")
print(f"  Point count matches:    "
      f"{'PASS' if collection_info.points_count == 9380 else 'FAIL'}")
print(f"  Metadata verified:      PASS")
print(f"  Vectors verified:       PASS")
print(f"  Basic search working:   PASS")
print(f"\nCarries into Day 3:")
print(f"  Qdrant collection with 9380 indexed vectors")
print(f"  Ready for full search testing and Week 3 comparison")

=== Day 2 Complete Summary ===

Collection:      financial_headlines
Status:          green
Points uploaded: 9380
Vector size:     384 dimensions
Distance:        Cosine

Verification:
  Point count matches:    PASS
  Metadata verified:      PASS
  Vectors verified:       PASS
  Basic search working:   PASS

Carries into Day 3:
  Qdrant collection with 9380 indexed vectors
  Ready for full search testing and Week 3 comparison


Day 3: Search Testing & Week 3 Comparison


In [1]:
import os
import json
import time
import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Filter,
    FieldCondition,
    MatchValue
)
from sentence_transformers import SentenceTransformer

COLLECTION_NAME = "financial_headlines"
VECTOR_SIZE     = 384
LABEL_MAP       = {0: "Bearish", 1: "Bullish", 2: "Neutral"}


client = QdrantClient(host="localhost", port=6333)


embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


embeddings_matrix = np.load("headline_embeddings.npy")
with open("metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("All imports and connections successful")
print(f"Qdrant collection: {COLLECTION_NAME}")
print(f"Local embeddings:  {embeddings_matrix.shape}")
print(f"Metadata entries:  {len(metadata)}")

c:\Users\RIDDHI ASHAR\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4223.17it/s]


All imports and connections successful
Qdrant collection: financial_headlines
Local embeddings:  (9380, 384)
Metadata entries:  9380


In [2]:
def qdrant_search(query, top_k=5, min_similarity=0.3,
                  sentiment_filter=None, ticker_filter=None):
    """
    Search Qdrant vector database for similar headlines.

    Args:
        query:            raw search text
        top_k:            number of results to return
        min_similarity:   minimum score threshold
        sentiment_filter: "Bearish", "Bullish", "Neutral" or None
        ticker_filter:    "$AAPL", "$TSLA" etc or None

    Returns:
        list of result dictionaries
    """
    if not query or not query.strip():
        raise ValueError("Query cannot be empty.")

    # encode query
    query_vector = embedding_model.encode(
        query, convert_to_numpy=True
    ).tolist()

    # build filters
    must_conditions = []
    if sentiment_filter:
        must_conditions.append(
            FieldCondition(
                key="sentiment",
                match=MatchValue(value=sentiment_filter)
            )
        )
    if ticker_filter:
        must_conditions.append(
            FieldCondition(
                key="ticker",
                match=MatchValue(value=ticker_filter)
            )
        )

    query_filter = Filter(must=must_conditions) \
        if must_conditions else None

    # search Qdrant
    response = client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        limit=top_k,
        with_payload=True,
        query_filter=query_filter,
        score_threshold=min_similarity
    )

    # build results
    results = []
    for rank, point in enumerate(response.points):
        results.append({
            "rank":       rank + 1,
            "similarity": float(point.score),
            "headline":   point.payload["original_text"],
            "cleaned":    point.payload["cleaned_text"],
            "sentiment":  point.payload["sentiment"],
            "ticker":     point.payload["ticker"],
            "id":         point.id
        })

    return results


def display_qdrant_results(query, results):
    """Display Qdrant search results cleanly."""
    print(f"Query: {query}")
    print(f"{'='*60}\n")
    if not results:
        print("No results found above similarity threshold.\n")
        return
    for r in results:
        print(f"Rank {r['rank']} — Score: {r['similarity']:.4f}")
        print(f"  Headline:  {r['headline'][:75]}")
        print(f"  Sentiment: {r['sentiment']}")
        print(f"  Ticker:    {r['ticker'] if r['ticker'] else 'None'}")
        print()

print("qdrant_search() defined successfully")

qdrant_search() defined successfully


In [3]:
query = "Tesla beats earnings expectations"
start = time.time()
results = qdrant_search(query, top_k=5, min_similarity=0.3)
latency = (time.time() - start) * 1000
print(f"Qdrant latency: {latency:.2f}ms")
display_qdrant_results(query, results)

Qdrant latency: 530.04ms
Query: Tesla beats earnings expectations

Rank 1 — Score: 0.6294
  Headline:  Tesla's stock surges 8.8% to top the $800 level
  Sentiment: Bullish
  Ticker:    None

Rank 2 — Score: 0.6158
  Headline:  Tesla's Return to Profitability Is Not Sustainable
  Sentiment: Bearish
  Ticker:    None

Rank 3 — Score: 0.5829
  Headline:  Tesla stock is wrapping up its best month since 2013, adding $40 billion in
  Sentiment: Bullish
  Ticker:    None

Rank 4 — Score: 0.5686
  Headline:  Money manager says she has 'very high' confidence Tesla shares reach $7,000
  Sentiment: Bullish
  Ticker:    None

Rank 5 — Score: 0.5446
  Headline:  $TSLA - Why Tesla Remains A Strong Buy. https://t.co/LyHcrFPlEH #markets #s
  Sentiment: Bullish
  Ticker:    $TSLA



In [4]:
query = "stock market crash recession fears"
start = time.time()
results = qdrant_search(query, top_k=5, min_similarity=0.3)
latency = (time.time() - start) * 1000
print(f"Qdrant latency: {latency:.2f}ms")
display_qdrant_results(query, results)

Qdrant latency: 140.62ms
Query: stock market crash recession fears

Rank 1 — Score: 0.7500
  Headline:  This economic trend could revive recession fears
  Sentiment: Bearish
  Ticker:    None

Rank 2 — Score: 0.6694
  Headline:  JUST RELEASED: Our newest #EconFocus looks at our fear of #recession. Could
  Sentiment: Neutral
  Ticker:    None

Rank 3 — Score: 0.6036
  Headline:  Risk of recession returns to Germany
  Sentiment: Bearish
  Ticker:    None

Rank 4 — Score: 0.5975
  Headline:  Small Business Optimism Crashes By Most Ever https://t.co/RWqy5LU6x8
  Sentiment: Bearish
  Ticker:    None

Rank 5 — Score: 0.5834
  Headline:  German Recession Risk Returns With Crisis-Era Plunge in Output
  Sentiment: Bearish
  Ticker:    None



In [5]:
query = "Federal Reserve interest rate decision inflation"
start = time.time()
results = qdrant_search(query, top_k=5, min_similarity=0.3)
latency = (time.time() - start) * 1000
print(f"Qdrant latency: {latency:.2f}ms")
display_qdrant_results(query, results)

Qdrant latency: 135.94ms
Query: Federal Reserve interest rate decision inflation

Rank 1 — Score: 0.6117
  Headline:  Read here for instant analysis of the Federal Reserve's interest rate decis
  Sentiment: Neutral
  Ticker:    None

Rank 2 — Score: 0.6027
  Headline:  MPC Decision: MPC Keeps Rates On Hold Amid ‘Highly Uncertain’ Inflation Out
  Sentiment: Neutral
  Ticker:    None

Rank 3 — Score: 0.5668
  Headline:  Inflation remains low and Fed Chairman Jerome Powell has as good as ruled o
  Sentiment: Neutral
  Ticker:    None

Rank 4 — Score: 0.5461
  Headline:  The Fed may lower the rate on its support facility for money market funds, 
  Sentiment: Neutral
  Ticker:    None

Rank 5 — Score: 0.5412
  Headline:  Fed's Mester, upbeat on outlook, say central bank can watch and wait on int
  Sentiment: Bullish
  Ticker:    None



In [6]:
query = "Apple iPhone revenue growth"
start = time.time()
results = qdrant_search(query, top_k=5, min_similarity=0.3)
latency = (time.time() - start) * 1000
print(f"Qdrant latency: {latency:.2f}ms")
display_qdrant_results(query, results)

Qdrant latency: 158.14ms
Query: Apple iPhone revenue growth

Rank 1 — Score: 0.6099
  Headline:  $AAPL - Apple forecasts 100M+ 5G iPhone sales - report https://t.co/QhkaVvX
  Sentiment: Bullish
  Ticker:    $AAPL

Rank 2 — Score: 0.5902
  Headline:  Apple forecasts 100M+ 5G iPhone sales - report
  Sentiment: Bullish
  Ticker:    None

Rank 3 — Score: 0.5580
  Headline:  Wearables Sales Surge, Bolstering Apple Stock
  Sentiment: Bullish
  Ticker:    None

Rank 4 — Score: 0.5502
  Headline:  Twitter Beats Revenue, User Growth Estimates in Fourth Quarter
  Sentiment: Bullish
  Ticker:    None

Rank 5 — Score: 0.5326
  Headline:  Dave & Buster's $PLAY reports earnings: 
Total revenues increased 6.1% to $
  Sentiment: Bullish
  Ticker:    $PLAY



In [7]:
query = "analyst raises price target strong buy rating"
start = time.time()
results = qdrant_search(query, top_k=5, min_similarity=0.3)
latency = (time.time() - start) * 1000
print(f"Qdrant latency: {latency:.2f}ms")
display_qdrant_results(query, results)

Qdrant latency: 108.31ms
Query: analyst raises price target strong buy rating

Rank 1 — Score: 0.5766
  Headline:  Target shares surge after company crushes earnings and raises forecast
  Sentiment: Bullish
  Ticker:    None

Rank 2 — Score: 0.5726
  Headline:  Earnings Update: Here's Why Analysts Just Lifted Their Milestone Scientific
  Sentiment: Bullish
  Ticker:    None

Rank 3 — Score: 0.5694
  Headline:  Best Buy stock price target raised to $105 from $86 at Oppenheimer
  Sentiment: Bullish
  Ticker:    None

Rank 4 — Score: 0.5661
  Headline:  Earnings Update: Here's Why Analysts Just Lifted Their Aemetis, Inc. Price 
  Sentiment: Bullish
  Ticker:    None

Rank 5 — Score: 0.5596
  Headline:  AMD Option Traders Turn Bullish Following Second Analyst Target Hike
  Sentiment: Bullish
  Ticker:    None



In [8]:
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine

def week3_search(query, top_k=5, min_similarity=0.3):
    """Week 3 brute force cosine similarity search."""
    query_vec = embedding_model.encode(
        query, convert_to_numpy=True
    ).reshape(1, -1)
    sims = sklearn_cosine(query_vec, embeddings_matrix)[0]
    top_indices = sims.argsort()[::-1]
    results = []
    for idx in top_indices:
        if len(results) >= top_k:
            break
        if sims[idx] < min_similarity:
            break
        results.append({
            "rank":       len(results) + 1,
            "similarity": float(sims[idx]),
            "headline":   metadata[idx]["original_text"],
            "sentiment":  metadata[idx]["sentiment"],
        })
    return results


with open("week3_search_baseline.json", "r") as f:
    baseline = json.load(f)

test_queries = [b["query"] for b in baseline]

print("=== Qdrant vs Week 3 Comparison ===\n")
print(f"{'Query':<40} {'Week3':>10} {'Qdrant':>10} {'Speedup':>10}")
print("-" * 72)

comparison_results = []
for query in test_queries:
   
    start = time.time()
    w3_results = week3_search(query, top_k=5)
    w3_latency = (time.time() - start) * 1000

    
    start = time.time()
    qd_results = qdrant_search(query, top_k=5)
    qd_latency = (time.time() - start) * 1000

    speedup = w3_latency / qd_latency if qd_latency > 0 else 0

    print(f"{query[:38]:<40} "
          f"{w3_latency:>8.1f}ms "
          f"{qd_latency:>8.1f}ms "
          f"{speedup:>8.1f}x")

    comparison_results.append({
        "query":         query,
        "week3_ms":      round(w3_latency, 2),
        "qdrant_ms":     round(qd_latency, 2),
        "speedup":       round(speedup, 2),
        "week3_top":     w3_results[0]["headline"][:60] if w3_results else "",
        "qdrant_top":    qd_results[0]["headline"][:60] if qd_results else "",
        "results_match": (
            w3_results[0]["headline"] == qd_results[0]["headline"]
            if w3_results and qd_results else False
        )
    })


with open("week4_comparison.json", "w") as f:
    json.dump(comparison_results, f, indent=2)

print(f"\nSaved: week4_comparison.json")

=== Qdrant vs Week 3 Comparison ===

Query                                         Week3     Qdrant    Speedup
------------------------------------------------------------------------
Tesla beats earnings expectations            57.1ms     84.8ms      0.7x
stock market crash recession fears           26.7ms     67.4ms      0.4x
Federal Reserve interest rate decision       35.9ms     61.5ms      0.6x
Apple iPhone revenue growth                  18.9ms     52.3ms      0.4x
analyst raises price target strong buy       38.6ms     69.7ms      0.6x

Saved: week4_comparison.json


In [9]:
print("=== Latency Analysis ===\n")

print("Why Qdrant appears slower for 9380 vectors:")
print("  HTTP round trip overhead: ~30-50ms per query")
print("  Numpy direct RAM access:  no overhead")
print("  At 9380 vectors both are fast — overhead dominates\n")

print("Why Qdrant is still the right choice:")
print("  Scales to millions of vectors efficiently")
print("  Persistent storage — data survives restarts")
print("  Sentiment and ticker filtering built in")
print("  Multiple clients can query simultaneously")
print("  FastAPI can query it directly in Week 5")
print("  Production ready — not a research prototype\n")

print("Expected Qdrant advantage at scale:")
print("  10K vectors:    numpy ~50ms    Qdrant ~70ms")
print("  100K vectors:   numpy ~500ms   Qdrant ~80ms")
print("  1M vectors:     numpy ~5000ms  Qdrant ~100ms")
print("  10M vectors:    numpy timeout  Qdrant ~120ms")

=== Latency Analysis ===

Why Qdrant appears slower for 9380 vectors:
  HTTP round trip overhead: ~30-50ms per query
  Numpy direct RAM access:  no overhead
  At 9380 vectors both are fast — overhead dominates

Why Qdrant is still the right choice:
  Scales to millions of vectors efficiently
  Persistent storage — data survives restarts
  Sentiment and ticker filtering built in
  Multiple clients can query simultaneously
  FastAPI can query it directly in Week 5
  Production ready — not a research prototype

Expected Qdrant advantage at scale:
  10K vectors:    numpy ~50ms    Qdrant ~70ms
  100K vectors:   numpy ~500ms   Qdrant ~80ms
  1M vectors:     numpy ~5000ms  Qdrant ~100ms
  10M vectors:    numpy timeout  Qdrant ~120ms


In [10]:
print("=== Detailed Comparison Analysis ===\n")

total_w3   = sum(r["week3_ms"]  for r in comparison_results)
total_qd   = sum(r["qdrant_ms"] for r in comparison_results)
avg_speedup = total_w3 / total_qd if total_qd > 0 else 0
results_match = sum(1 for r in comparison_results if r["results_match"])

print(f"Average Week 3 latency:   {total_w3/len(comparison_results):.1f}ms")
print(f"Average Qdrant latency:   {total_qd/len(comparison_results):.1f}ms")
print(f"Average speedup:          {avg_speedup:.1f}x faster")
print(f"Top result matches:       {results_match}/{len(comparison_results)}")

print(f"\nResult quality check:")
for r in comparison_results:
    match = "MATCH" if r["results_match"] else "DIFF"
    print(f"  [{match}] {r['query'][:45]}")
    if not r["results_match"]:
        print(f"    Week3:  {r['week3_top'][:55]}")
        print(f"    Qdrant: {r['qdrant_top'][:55]}")

=== Detailed Comparison Analysis ===

Average Week 3 latency:   35.5ms
Average Qdrant latency:   67.1ms
Average speedup:          0.5x faster
Top result matches:       5/5

Result quality check:
  [MATCH] Tesla beats earnings expectations
  [MATCH] stock market crash recession fears
  [MATCH] Federal Reserve interest rate decision
  [MATCH] Apple iPhone revenue growth
  [MATCH] analyst raises price target strong buy


In [11]:
print("=== Week 4 Unit Tests ===\n")

passed = 0
failed = 0

def run_test(name, condition):
    global passed, failed
    if condition:
        print(f"  [PASS] {name}")
        passed += 1
    else:
        print(f"  [FAIL] {name}")
        failed += 1

collections = client.get_collections()
collection_names = [c.name for c in collections.collections]
run_test(
    "Collection financial_headlines exists",
    COLLECTION_NAME in collection_names
)


info = client.get_collection(COLLECTION_NAME)
run_test(
    "Collection has 9380 points",
    info.points_count == 9380
)


results = qdrant_search("Tesla beats earnings", top_k=5)
run_test(
    "Basic search returns results",
    len(results) > 0
)

run_test(
    "Search results have correct keys",
    all(k in results[0] for k in
        ["rank", "similarity", "headline", "sentiment", "ticker"])
)

sims = [r["similarity"] for r in results]
run_test(
    "Results sorted by similarity descending",
    all(sims[i] >= sims[i+1] for i in range(len(sims)-1))
)

run_test(
    "All results above min_similarity threshold",
    all(r["similarity"] >= 0.3 for r in results)
)


bearish_results = qdrant_search(
    "stock falls recession",
    top_k=5,
    sentiment_filter="Bearish"
)
run_test(
    "Sentiment filter returns only Bearish results",
    all(r["sentiment"] == "Bearish" for r in bearish_results)
)

bullish_results = qdrant_search(
    "company beats earnings",
    top_k=5,
    sentiment_filter="Bullish"
)
run_test(
    "Sentiment filter returns only Bullish results",
    all(r["sentiment"] == "Bullish" for r in bullish_results)
)

# Test 9 — empty query raises error
try:
    qdrant_search("", top_k=5)
    run_test("Empty query raises ValueError", False)
except ValueError:
    run_test("Empty query raises ValueError", True)

# Test 10 — high threshold returns fewer results
results_low  = qdrant_search("Tesla", top_k=10, min_similarity=0.1)
results_high = qdrant_search("Tesla", top_k=10, min_similarity=0.8)
run_test(
    "Higher threshold returns fewer or equal results",
    len(results_high) <= len(results_low)
)

# Test 11 — metadata correctly stored
point = client.retrieve(
    collection_name=COLLECTION_NAME,
    ids=[0],
    with_payload=True
)[0]
run_test(
    "Metadata fields stored correctly",
    all(k in point.payload for k in
        ["original_text", "sentiment", "ticker", "label"])
)

# Test 12 — vector correctly stored
point_with_vec = client.retrieve(
    collection_name=COLLECTION_NAME,
    ids=[0],
    with_vectors=True
)[0]
run_test(
    "Vector has correct dimensions",
    len(point_with_vec.vector) == 384
)

print(f"\n{'='*40}")
print(f"Results: {passed} passed, {failed} failed")
if failed == 0:
    print("All unit tests passed ✅")
    print("Week 4 fully verified")
else:
    print(f"{failed} test(s) failed")

=== Week 4 Unit Tests ===

  [PASS] Collection financial_headlines exists
  [PASS] Collection has 9380 points
  [PASS] Basic search returns results
  [PASS] Search results have correct keys
  [PASS] Results sorted by similarity descending
  [PASS] All results above min_similarity threshold
  [PASS] Sentiment filter returns only Bearish results
  [PASS] Sentiment filter returns only Bullish results
  [PASS] Empty query raises ValueError
  [PASS] Higher threshold returns fewer or equal results
  [PASS] Metadata fields stored correctly
  [PASS] Vector has correct dimensions

Results: 12 passed, 0 failed
All unit tests passed ✅
Week 4 fully verified


In [14]:
files = [
    "vector_db.py",
    "week4_comparison.json",
    "headline_embeddings.npy",
    "metadata.json",
    "embedding_pipeline.py",
    "semantic_search.py",
    "metadata_pipeline.py",
    "week3_search_baseline.json",
]

print("=== Week 4 Final File Check ===\n")
for file in files:
    exists = os.path.exists(file)
    size = os.path.getsize(file) if exists else 0
    print(f"{'EXISTS' if exists else 'MISSING'} — {file} ({size:,} bytes)")

=== Week 4 Final File Check ===

EXISTS — vector_db.py (4,566 bytes)
EXISTS — week4_comparison.json (1,581 bytes)
EXISTS — headline_embeddings.npy (14,407,808 bytes)
EXISTS — metadata.json (3,545,127 bytes)
EXISTS — embedding_pipeline.py (3,444 bytes)
EXISTS — semantic_search.py (2,622 bytes)
EXISTS — metadata_pipeline.py (4,718 bytes)
EXISTS — week3_search_baseline.json (1,181 bytes)


In [13]:
vector_db_content = '''import os
import json
import numpy as np
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct,
    Filter, FieldCondition, MatchValue
)
from sentence_transformers import SentenceTransformer

COLLECTION_NAME = "financial_headlines"
VECTOR_SIZE     = 384
BATCH_SIZE      = 100
MODEL_NAME      = "all-MiniLM-L6-v2"

print(f"Loading embedding model: {MODEL_NAME}")
embedding_model = SentenceTransformer(MODEL_NAME)
print(f"Model loaded — {embedding_model.get_embedding_dimension()} dims")


def get_client(host="localhost", port=6333):
    """Connect to Qdrant instance."""
    client = QdrantClient(host=host, port=port)
    print(f"Connected to Qdrant at {host}:{port}")
    return client


def create_collection(client, collection_name=COLLECTION_NAME,
                      vector_size=VECTOR_SIZE):
    """Create vector collection in Qdrant."""
    try:
        client.delete_collection(collection_name)
    except Exception:
        pass
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(
            size=vector_size,
            distance=Distance.COSINE
        )
    )
    print(f"Collection created: {collection_name}")


def upload_embeddings(client, embeddings, metadata,
                      collection_name=COLLECTION_NAME,
                      batch_size=BATCH_SIZE):
    """Upload embeddings and metadata to Qdrant."""
    total = len(metadata)
    uploaded = 0
    for i in range(0, total, batch_size):
        batch_emb  = embeddings[i:i + batch_size]
        batch_meta = metadata[i:i + batch_size]
        points = [
            PointStruct(
                id=m["id"],
                vector=e.tolist(),
                payload={
                    "original_text": m["original_text"],
                    "cleaned_text":  m["cleaned_text"],
                    "ticker":        m["ticker"],
                    "tickers_list":  m["tickers_list"],
                    "label":         m["label"],
                    "sentiment":     m["sentiment"],
                    "word_count":    m["word_count"],
                    "token_count":   m["token_count"]
                }
            )
            for e, m in zip(batch_emb, batch_meta)
        ]
        client.upsert(collection_name=collection_name, points=points)
        uploaded += len(points)
    print(f"Uploaded {uploaded} points to {collection_name}")
    return uploaded


def qdrant_search(client, query, top_k=5,
                  min_similarity=0.3,
                  sentiment_filter=None,
                  ticker_filter=None,
                  collection_name=COLLECTION_NAME):
    """Search Qdrant for similar headlines."""
    if not query or not query.strip():
        raise ValueError("Query cannot be empty.")

    query_vector = embedding_model.encode(
        query, convert_to_numpy=True
    ).tolist()

    must_conditions = []
    if sentiment_filter:
        must_conditions.append(
            FieldCondition(
                key="sentiment",
                match=MatchValue(value=sentiment_filter)
            )
        )
    if ticker_filter:
        must_conditions.append(
            FieldCondition(
                key="ticker",
                match=MatchValue(value=ticker_filter)
            )
        )

    query_filter = Filter(must=must_conditions) \\
        if must_conditions else None

    response = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=top_k,
        with_payload=True,
        query_filter=query_filter,
        score_threshold=min_similarity
    )

    return [
        {
            "rank":       rank + 1,
            "similarity": float(p.score),
            "headline":   p.payload["original_text"],
            "cleaned":    p.payload["cleaned_text"],
            "sentiment":  p.payload["sentiment"],
            "ticker":     p.payload["ticker"],
            "id":         p.id
        }
        for rank, p in enumerate(response.points)
    ]


if __name__ == "__main__":
    client = get_client()
    results = qdrant_search(
        client,
        "Tesla beats earnings expectations",
        top_k=3
    )
    print("\\nTest search results:")
    for r in results:
        print(f"  {r[\'rank\']}. [{r[\'sentiment\']}] "
              f"(sim={r[\'similarity\']:.3f}) "
              f"{r[\'headline\'][:60]}")
'''

with open("vector_db.py", "w", encoding="utf-8") as f:
    f.write(vector_db_content)

print("Saved: vector_db.py")
print(f"File size: {os.path.getsize('vector_db.py'):,} bytes")

Saved: vector_db.py
File size: 4,566 bytes
